# Project Template

Copy this repository to get started with the project

## Setup

Modify the following 2 cells to import libraries and helper files that you might need for your project

In [1]:
# Some possible helper files from class

!curl -O -L -s https://github.com/PSAM-5020-2026S-A/5020-utils/raw/main/src/audio_utils.py
!curl -O -L -s https://github.com/PSAM-5020-2026S-A/5020-utils/raw/main/src/data_utils.py
!curl -O -L -s https://github.com/PSAM-5020-2026S-A/5020-utils/raw/main/src/image_utils.py
!curl -O -L -s https://github.com/PSAM-5020-2026S-A/5020-utils/raw/main/src/text_utils.py

In [2]:
# Some possible libraries.
# This isn't complete.

import librosa
import matplotlib.pyplot as plt
import pandas as pd
import PIL.Image as PImage
import numpy as np

from os import listdir, path

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC

from audio_utils import fft, stft
from data_utils import object_from_json_url, classification_error, display_confusion_matrix
from image_utils import get_pixels, make_image
from text_utils import get_top_words

## Milestone 01

### Short Description... maybe... ?

In [ ]:
#Cleaning and Set up

audio_files = {
    "Caribbean": "assets/Whale_carrinbean_Sep2015.wav",
    "Australia": "assets/Whale_Queens_Aus_Aug2017.wav",
    "Santa Cruz": "assets/Whale_Santa_Cruz_Dec2016.wav"
}

clean_audio_data = {}
standard_sr = 22050

for region, file_path in audio_files.items():
    y_raw, sr = librosa.load(file_path, sr=standard_sr)
    
    y_filtered = librosa.effects.preemphasis(y_raw)    
    y_trimmed, index = librosa.effects.trim(y_filtered, top_db=20)
    y_normalized = y_trimmed / np.max(np.abs(y_trimmed))
    
    #new dictionary
    clean_audio_data[region] = y_normalized


In [16]:
#Feature Extraction

extracted_regions = []
extracted_centroids = []
extracted_mfccs = []

#slicing
chunk_duration = 3.0 # seconds
samples_per_chunk = int(chunk_duration * standard_sr)



for region, y_clean in clean_audio_data.items():
    
    #slicing
    for i in range(0, len(y_clean), samples_per_chunk):
        chunk = y_clean[i : i + samples_per_chunk]
        
        
        if len(chunk) < samples_per_chunk:
            continue
            
        #Freq
        centroid = librosa.feature.spectral_centroid(y=chunk, sr=standard_sr)
        mean_centroid = np.mean(centroid) # Average it out for the chunk
        
        # Timbre MFCC
        mfccs = librosa.feature.mfcc(y=chunk, sr=standard_sr, n_mfcc=13)
        mean_mfccs = np.mean(mfccs.T, axis=0) # Average the 13 texture numbers
        
        #MAIN
        extracted_regions.append(region)
        extracted_centroids.append(mean_centroid)
        extracted_mfccs.append(mean_mfccs)